# Free Odds: The One Bet with Zero House Edge

Free odds (taken on the pass/come side, *laid* on the don't side)
are the only casino wager with **zero** expected value. They pay
true odds — the exact ratio that compensates the per-point win
probability — so the house makes nothing on them per dollar
wagered.

That doesn't mean free odds make the craps player magically
profitable. The *line* bet they attach to still has its original
1.414% (pass) or 1.364% (don't pass) edge. What odds do is
**dilute** that edge across a larger total wager: every extra
dollar of zero-EV odds shrinks the composite edge proportionally.

This notebook derives the zero-EV claim in exact `Fraction`
arithmetic, writes out the composite edge as a function of the
odds policy, plots the dilution curve for the standard multipliers
(1x, 2x, 3x, 3-4-5x, 5x, 10x, 100x), and validates the 3-4-5x
result with a seeded Monte Carlo.

## Zero expected value — the cancellation

A pass-odds bet on point $p$ pays $c_7 / c_p$ per \$1 wagered if
the point is rolled before 7, where $c_p = \mathrm{count}(p)$ is
the number of two-dice outcomes summing to $p$ and $c_7 = 6$.

The win and loss probabilities are

$$
P(p \text{ before } 7) \;=\; \frac{c_p}{c_p + c_7}, \qquad
P(7 \text{ before } p) \;=\; \frac{c_7}{c_p + c_7}.
$$

Expected P/L per \$1 wagered:

$$
E[P/L] \;=\; \frac{c_p}{c_p + c_7} \cdot \frac{c_7}{c_p} \;+\; \frac{c_7}{c_p + c_7} \cdot (-1)
       \;=\; \frac{c_7}{c_p + c_7} - \frac{c_7}{c_p + c_7}
       \;=\; 0.
$$

The $c_p$ factors cancel in the win term. This works for *every*
point, so the expected value of a pass-odds bet is identically
zero — not numerically close, exactly zero as a `Fraction`.

Lay odds mirror the argument: payout $c_p / c_7$ instead of
$c_7 / c_p$, and the win/loss probabilities swap, so the same
cancellation gives zero. Pass and lay are two sides of the same
fair bet.

In [ ]:
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np

from craps_lab.bets import POINT_NUMBERS
from craps_lab.probability import (
    LAY_3_4_5X,
    ODDS_3_4_5X,
    dont_pass_house_edge,
    dont_pass_lay_odds_expected_value,
    dont_pass_plus_lay_odds_edge,
    pass_line_house_edge,
    pass_line_plus_odds_edge,
    pass_odds_expected_value,
    uniform_odds_policy,
)

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Pass odds expected value per point:")
for point in POINT_NUMBERS:
    ev = pass_odds_expected_value(point)
    print(f"  point {point:2d}: E[P/L] = {ev}")

print()
print("Lay odds expected value per point:")
for point in POINT_NUMBERS:
    ev = dont_pass_lay_odds_expected_value(point)
    print(f"  point {point:2d}: E[P/L] = {ev}")

## Composite edge dilution

Expected loss per round is *unchanged* when you add odds — they're
zero-EV. What changes is the **denominator** in "house edge per
dollar wagered". If $E[\text{odds}]$ is the average odds bet per
round (averaging over the 12/36 come-out rounds with no point
established and the 24/36 rounds where a point is set), the
composite edge is

$$
\text{edge}_{\text{composite}} \;=\; \frac{\text{base edge}}{1 + E[\text{odds}]}.
$$

For uniform $k$x pass-odds, $E[\text{odds}] = k \cdot 24/36 = 2k/3$, so the edge dilutes as

$$
\text{edge}(k) \;=\; \frac{7/495}{1 + 2k/3}.
$$

For the canonical 3-4-5x policy, $E[\text{odds}] = (9 + 16 + 25 + 25 + 16 + 9)/36 = 100/36 = 25/9$ and the composite edge is $(7/495)/(34/9) = 7/1870 \approx 0.374\%$.

An analogous formula gives the don't pass + lay edge.

In [ ]:
multipliers = [0, 1, 2, 3, 5, 10, 100]

pass_edges = [
    float(pass_line_plus_odds_edge(uniform_odds_policy(m))) for m in multipliers
]
dont_edges = [
    float(dont_pass_plus_lay_odds_edge(uniform_odds_policy(m))) for m in multipliers
]

pass_3_4_5 = float(pass_line_plus_odds_edge(ODDS_3_4_5X))
dont_3_4_5 = float(dont_pass_plus_lay_odds_edge(LAY_3_4_5X))

fig, ax = plt.subplots()
ax.plot(multipliers, pass_edges, "o-", color="steelblue", linewidth=2, label="pass line + $k$x odds")
ax.plot(multipliers, dont_edges, "o-", color="tomato", linewidth=2, label="don't pass + $k$x lay")
ax.axhline(
    pass_3_4_5,
    color="steelblue",
    linestyle="--",
    alpha=0.6,
    label=f"pass + 3-4-5x ≈ {pass_3_4_5 * 100:.3f}%",
)
ax.axhline(
    dont_3_4_5,
    color="tomato",
    linestyle="--",
    alpha=0.6,
    label=f"don't + 3-4-5x equiv ≈ {dont_3_4_5 * 100:.3f}%",
)
ax.set_xscale("symlog", linthresh=1)
ax.set_xlabel("odds multiplier $k$")
ax.set_ylabel("composite house edge")
ax.set_title("Composite edge vs. odds multiplier")
ax.legend(loc="upper right")
ax.set_ylim(0, 0.016)
plt.tight_layout()
plt.show()

print()
print(f"{'multiplier':<14}{'pass edge':<16}{'dont edge':<16}")
print("-" * 46)
for m, p, d in zip(multipliers, pass_edges, dont_edges, strict=True):
    print(f"{m}x{' ':<12}{p * 100:>7.4f}%      {d * 100:>7.4f}%")
print(f"3-4-5x{' ':<8}{pass_3_4_5 * 100:>7.4f}%      {dont_3_4_5 * 100:>7.4f}%")

## What the curve shows

1. **Both curves converge toward zero** as the multiplier grows,
   but they never actually reach it. The composite edge is
   $\text{base}/(1 + E[\text{odds}])$, which is asymptotically
   proportional to $1/k$ for large $k$, not identically zero.
2. **Don't pass + lay is always below pass + take** at the same
   multiplier, and dramatically so: at 10x it's ~0.11% vs ~0.18%.
   The reason is the smaller base edge (3/220 vs 7/495) gets
   compounded by a larger denominator, because 10x lay is bigger
   than the average 10x take on a "weighted by point probability"
   basis. (At the 3-4-5x comparison, the same asymmetry appears:
   uniform 6x lay vs. 25/9 ≈ 2.78x average take.)
3. **100x odds** is real: a few Las Vegas casinos offer it to
   high-rollers. At 100x, the pass composite edge is ~0.02%,
   essentially a rounding error away from break-even.

## Seeded Monte Carlo validation

As a final check, simulate 200,000 games of pass line + 3-4-5x
odds and don't pass + 3-4-5x lay on the seeded `DiceRoller`,
tracking total P/L and total wagered per game. The empirical
composite edge should land within a few tenths of a percentage
point of the analytical `Fraction` value. (The same cross-check
runs in `tests/test_convergence.py` with a 0.002 absolute
tolerance as a CI guard.)

In [ ]:
from craps_lab.bets import (
    DONT_PASS_BAR,
    DONT_PASS_COME_OUT_LOSES,
    DONT_PASS_COME_OUT_WINS,
    DONT_PASS_LAY_PAYOUT_RATIO,
    PASS_LINE_CRAPS_LOSERS,
    PASS_LINE_NATURAL_WINNERS,
    PASS_ODDS_PAYOUT_RATIO,
    SEVEN,
)
from craps_lab.dice import DiceRoller


def simulate_pass_with_odds(n_games, seed, odds_policy):
    roller = DiceRoller(seed=seed)
    total_pl = 0.0
    total_wagered = 0.0
    for _ in range(n_games):
        total_wagered += 1.0
        come_out = roller.roll().total
        if come_out in PASS_LINE_NATURAL_WINNERS:
            total_pl += 1.0
            continue
        if come_out in PASS_LINE_CRAPS_LOSERS:
            total_pl -= 1.0
            continue
        point = come_out
        odds_bet = float(odds_policy[point])
        odds_payout = float(PASS_ODDS_PAYOUT_RATIO[point])
        total_wagered += odds_bet
        while True:
            roll = roller.roll().total
            if roll == SEVEN:
                total_pl -= 1.0 + odds_bet
                break
            if roll == point:
                total_pl += 1.0 + odds_bet * odds_payout
                break
    return -total_pl / total_wagered


def simulate_dont_pass_with_lay(n_games, seed, lay_policy):
    roller = DiceRoller(seed=seed)
    total_pl = 0.0
    total_wagered = 0.0
    for _ in range(n_games):
        total_wagered += 1.0
        come_out = roller.roll().total
        if come_out in DONT_PASS_COME_OUT_WINS:
            total_pl += 1.0
            continue
        if come_out in DONT_PASS_COME_OUT_LOSES:
            total_pl -= 1.0
            continue
        if come_out == DONT_PASS_BAR:
            continue
        point = come_out
        lay_bet = float(lay_policy[point])
        lay_payout = float(DONT_PASS_LAY_PAYOUT_RATIO[point])
        total_wagered += lay_bet
        while True:
            roll = roller.roll().total
            if roll == SEVEN:
                total_pl += 1.0 + lay_bet * lay_payout
                break
            if roll == point:
                total_pl -= 1.0 + lay_bet
                break
    return -total_pl / total_wagered


N_GAMES = 200_000
SEED = 0xF00D

pass_empirical = simulate_pass_with_odds(N_GAMES, SEED, ODDS_3_4_5X)
dont_empirical = simulate_dont_pass_with_lay(N_GAMES, SEED + 1, LAY_3_4_5X)

print(f"pass + 3-4-5x:       empirical = {pass_empirical * 100:.4f}%,"
      f" analytical = {pass_3_4_5 * 100:.4f}%,"
      f" delta = {(pass_empirical - pass_3_4_5) * 100:+.4f}%")
print(f"don't + 3-4-5x lay:  empirical = {dont_empirical * 100:.4f}%,"
      f" analytical = {dont_3_4_5 * 100:.4f}%,"
      f" delta = {(dont_empirical - dont_3_4_5) * 100:+.4f}%")

## Takeaways

1. **Free odds are the only casino wager with zero expected value.**
   The payout ratios exactly compensate the per-point win
   probabilities, so $E[P/L]$ on a \$1 odds bet is **identically**
   zero as a `Fraction` — not rounded, not approximated, exactly
   zero.
2. **Composite edge dilutes as $\text{base} / (1 + E[\text{odds}])$.**
   Adding odds doesn't change your expected loss per round; it
   spreads that loss over a larger total wager, so the "edge per
   dollar" shrinks.
3. **At 3-4-5x**: pass line drops from $1.414\%$ to **$\approx 0.374\%$**, and don't pass drops from $1.364\%$ to **$\approx 0.273\%$**. Don't pass is *more* dilutable under the equivalent lay because uniform 6x lay is larger than the $\approx 2.78$x average take under 3-4-5x.
4. **At 100x odds** (real, offered by a few Vegas casinos), pass
   composite $\approx 0.02\%$: essentially a rounding error away
   from break-even.
5. **The Monte Carlo confirms the analytical values** within a
   few hundredths of a percent over 200,000 games. The unit test
   in `tests/test_convergence.py` pins this down with a 0.002
   absolute tolerance.

**Next up** in Phase 4: the craps **table state machine** — a
proper per-bet resolver that handles multiple simultaneous bets,
come-bet travel, and session bankroll tracking. Once that lands,
we can finally run full *sessions* instead of one-bet-at-a-time
Monte Carlos.